# 3.2.5 人脸AI智能检测系统交互流程设计

## 任务说明（摘自考试预览）

在安防监控、智能交通等领域，实时准确的人脸检测需求日益增长。传统的人脸检测方法在面对复杂光照、多角度、遮挡等情况时，检测效果往往不尽人意。随着深度学习技术的发展，基于深度神经网络的人脸检测模型展现出强大的性能优势。ONNX（Open Neural Network Exchange）作为一种开放式的神经网络交换格式，能够实现不同深度学习框架间的模型转换与共享，使得基于 ONNX 的人脸检测模型可以在多种环境下高效运行。本系统所使用的 “version-RFB-320.onnx” 模型，通过大量数据训练，能够快速准确地检测出图像中的人脸，在实际应用场景中具有重要价值。

AI 模型说明：“version-RFB-320.onnx” 模型是用于人脸检测的 ONNX 格式模型，对应的类别标签文件为 “voc-model-labels.txt” 。该模型的使用交互流程为：

（1）加载 “version-RFB-320.onnx” 模型和 “voc-model-labels.txt” 类别标签；

（2）加载本地测试图片文件夹 “imgs” 中的所有图片，并对每张图片进行预处理以符合模型输入要求；

（3）使用 “version-RFB-320.onnx” 模型对加载的图片进行人脸检测；

（4）在图片上绘制检测到的人脸框，并将处理后的图片保存到 “./detect_imgs_results_onnx” 文件夹中；

（5）统计所有图片中检测到的人脸总数并输出。

你作为一名人工智能训练师，请完成以下工作任务：

（1）补全该模型的使用交互流程对应的 Python 代码（3.2.5.ipynb），实现本地测试图片文件夹 “imgs” 中所有图片的人脸检测，将运行结果截图保存到3.2.5-1.jpg中，并将检测结果的图片上传。

（2）在上面的使用交互流程基础上，结合实际应用场景，设计在人脸检测系统中使用 “version-RFB-320.onnx” 模型的一种人机交互优化方案，包括交互界面布局、操作流程等内容，将其保存为docx文件，命名为 3.2.5.docx。

所有结果文件储存在桌面新建的考生文件夹中，文件夹命名为 “准考证号 + 身份证号后六位”。

---

以下为**刷题参考模板**：含完整可运行代码（编程题）或答题示例与生成 docx 草稿代码（主观题）。

## 人脸检测 ONNX（考场目录需含 vision 包与模型）

In [ ]:
import os
import time
import cv2
import numpy as np
import vision.utils.box_utils_numpy as box_utils
import onnxruntime as ort


def predict(width, height, confidences, boxes, prob_threshold, iou_threshold=0.3, top_k=-1):
    boxes = boxes[0]
    confidences = confidences[0]
    picked_box_probs = []
    picked_labels = []
    for class_index in range(1, confidences.shape[1]):
        probs = confidences[:, class_index]
        mask = probs > prob_threshold
        probs = probs[mask]
        if probs.shape[0] == 0:
            continue
        subset_boxes = boxes[mask, :]
        box_probs = np.concatenate(
            [subset_boxes, probs.reshape(-1, 1)], axis=1
        )
        box_probs = box_utils.hard_nms(
            box_probs, iou_threshold=iou_threshold, top_k=top_k
        )
        picked_box_probs.append(box_probs)
        picked_labels.extend([class_index] * box_probs.shape[0])
    if not picked_box_probs:
        return np.array([]), np.array([]), np.array([])
    picked_box_probs = np.concatenate(picked_box_probs)
    picked_box_probs[:, 0] *= width
    picked_box_probs[:, 1] *= height
    picked_box_probs[:, 2] *= width
    picked_box_probs[:, 3] *= height
    return (
        picked_box_probs[:, :4].astype(np.int32),
        np.array(picked_labels),
        picked_box_probs[:, 4],
    )


class_names = [
    name.strip() for name in open('voc-model-labels.txt').readlines()
]

ort_session = ort.InferenceSession(
    'version-RFB-320.onnx', providers=['CPUExecutionProvider']
)
input_name = ort_session.get_inputs()[0].name

result_path = './detect_imgs_results_onnx'
threshold = 0.7
path = 'imgs'
total_boxes = 0

if not os.path.exists(result_path):
    os.makedirs(result_path, exist_ok=True)

for file_path in os.listdir(path):
    img_path = os.path.join(path, file_path)
    orig_image = cv2.imread(img_path)
    if orig_image is None:
        continue
    image = cv2.cvtColor(orig_image, cv2.COLOR_BGR2RGB)
    image = cv2.resize(image, (320, 240))
    image_mean = np.array([127, 127, 127])
    image = (image - image_mean) / 128
    image = np.transpose(image, [2, 0, 1])
    image = np.expand_dims(image, axis=0)
    image = image.astype(np.float32)

    t0 = time.time()
    confidences, boxes = ort_session.run(None, {input_name: image})
    print("cost time:{}".format(time.time() - t0))

    boxes, labels, probs = predict(
        orig_image.shape[1],
        orig_image.shape[0],
        confidences,
        boxes,
        threshold,
    )
    for i in range(boxes.shape[0]):
        box = boxes[i, :]
        label = f'{class_names[labels[i]]}: {probs[i]:.2f}'
        cv2.rectangle(
            orig_image,
            (box[0], box[1]),
            (box[2], box[3]),
            (255, 255, 0),
            4,
        )
        cv2.imwrite(os.path.join(result_path, file_path), orig_image)
    total_boxes += boxes.shape[0]

print("sum:{}".format(total_boxes))

## 学习提示

- 编程题：请在**本案例文件夹**下运行 Notebook，以便相对路径 `data/...` 正确。
- 主观题：示例答案仅供参考，可按考场要求改写；若未安装 `python-docx`，可先 `pip install python-docx`。
- 交卷前核对平台要求的截图、HTML、docx 命名与保存路径。